In [1]:
# If not using colab, please comment this whole block:
# To get started, upload the Data_zip from github to google colab
!unzip -q /content/Data_zip.zip -d /content/

training_labels_dir = '/content/Data/training_labels'
training_images_dir = '/content/Data/training_images'
test_labels_dir = '/content/Data/test_labels'
test_images_dir = '/content/Data/test_images'

In [3]:
import pandas as pd
import os

training_labels = pd.read_csv(os.path.join(training_labels_dir, 'train.csv'))
training_labels['Id'] = training_labels['Id'].astype(str) + '.png'

In [4]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator

datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split = 0.2
)

train_generator = datagen.flow_from_dataframe(
    dataframe=training_labels,
    directory=training_images_dir,
    x_col='Id',
    y_col='Category',
    target_size=(32,32),
    color_mode='grayscale',
    class_mode='raw',
    batch_size=32
)

val_generator = datagen.flow_from_dataframe(
    dataframe=training_labels,
    directory=training_images_dir,
    x_col='Id',
    y_col='Category',
    subset='validation',             # Specify this is the validation set
    target_size=(32, 32),
    color_mode='grayscale',
    class_mode='raw',
    batch_size=32
)


model = models.Sequential([
    # Input layer + 1st Convolutional Block
    layers.Input(shape=(32, 32, 1)), # 32x32 size, 1 channel (grayscale)
    layers.Conv2D(16, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),

    # 2nd Convolutional Block
    layers.Conv2D(32, (3, 3), activation='relu', padding='same'),
    layers.MaxPooling2D((2, 2)),

    # Flatten the 2D images into a 1D array for the dense layers
    layers.Flatten(),

    # Fully connected layers
    layers.Dense(128, activation='relu'),

    # Output layer: 10 neurons for digits 0-9. No activation here because
    # we'll handle it in the loss function.
    layers.Dense(10)
])

model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

# Train the model!
history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10 # Start with 10 passes through the data
)

Found 17000 validated image filenames.
Found 3400 validated image filenames.
Epoch 1/10
532/532 ━━━━━━━━━━━━━━━━━━━━ 13s 17ms/step - accuracy: 0.9359 - loss: 0.2071 - val_accuracy: 0.9894 - val_loss: 0.0415
Epoch 2/10
532/532 ━━━━━━━━━━━━━━━━━━━━ 6s 12ms/step - accuracy: 0.9888 - loss: 0.0371 - val_accuracy: 0.9968 - val_loss: 0.0173
Epoch 3/10
532/532 ━━━━━━━━━━━━━━━━━━━━ 7s 13ms/step - accuracy: 0.9931 - loss: 0.0232 - val_accuracy: 0.9953 - val_loss: 0.0127
Epoch 4/10
532/532 ━━━━━━━━━━━━━━━━━━━━ 6s 12ms/step - accuracy: 0.9955 - loss: 0.0157 - val_accuracy: 0.9968 - val_loss: 0.0109
Epoch 5/10
532/532 ━━━━━━━━━━━━━━━━━━━━ 7s 13ms/step - accuracy: 0.9968 - loss: 0.0102 - val_accuracy: 0.9979 - val_loss: 0.0086
Epoch 6/10
532/532 ━━━━━━━━━━━━━━━━━━━━ 7s 12ms/step - accuracy: 0.9980 - loss: 0.0060 - val_accuracy: 0.9997 - val_loss: 0.0024
Epoch 7/10
532/532 ━━━━━━━━━━━━━━━━━━━━ 6s 12ms/step - accuracy: 0.9961 - loss: 0.0120 - val_accuracy: 0.9982 - val_loss: 0.0042
Epoch 8/10
532/532 

In [8]:
test_df = pd.read_csv(os.path.join(test_labels_dir, 'test.csv'))
test_df['Id'] = test_df['Id'].astype(str) + '.png'
test_df

,Id
0,56604.png
1,29396.png
2,43803.png
3,12313.png
4,10341.png
...,...
2995,9964.png
2996,67116.png
2997,7017.png
2998,41092.png


In [9]:
import numpy as np
import pandas as pd
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# 1. Load the Test CSV
# Make sure this path matches your test labels folder
test_df = pd.read_csv(os.path.join(test_labels_dir, 'test.csv'))

# IMPORTANT: Save the original integer IDs for the final submission
original_ids = test_df['Id'].copy()

# Fix the Kaggle Trap so Keras can find the images
test_df['Id'] = test_df['Id'].astype(str) + '.png'

# 2. Setup the Test Generator
test_datagen = ImageDataGenerator(rescale=1./255)

test_generator = test_datagen.flow_from_dataframe(
    dataframe=test_df,
    directory='/content/Data/test_images/', # Your local Colab test images folder
    x_col='Id',
    y_col=None,                        # There are no labels!
    target_size=(32, 32),
    color_mode='grayscale',
    class_mode=None,                   # Tells Keras not to expect a y_col
    batch_size=32,
    shuffle=False                      # CRUCIAL: Keeps predictions aligned with the CSV
)

# 3. Generate Predictions
print("Generating predictions...")
# model.predict gives us a matrix of probabilities for all 10 classes
raw_predictions = model.predict(test_generator)

# np.argmax looks at the probabilities for each image and picks the index (class)
# with the highest probability.
predicted_classes = np.argmax(raw_predictions, axis=1)

# 4. Create the Submission File
# Check your Kaggle sample_submission.csv. Usually the columns are 'Id' and whatever the target is called
submission_df = pd.DataFrame({
    'Id': original_ids,          # The original IDs without '.png'
    'category': predicted_classes  # Replace 'category' with the exact column name Kaggle expects
})

# Save to a CSV file without the pandas index column
submission_df.to_csv('submission.csv', index=False)
print("Saved as submission.csv! You can now download this and submit to Kaggle.")

Found 3000 validated image filenames.
Generating predictions...
94/94 ━━━━━━━━━━━━━━━━━━━━ 2s 18ms/step
Saved as submission.csv! You can now download this and submit to Kaggle.
